In [3]:
pip install librosa numpy pandas scikit-learn tensorflow matplotlib

Note: you may need to restart the kernel to use updated packages.


In [4]:
# pip install librosa numpy pandas scikit-learn tensorflow matplotlib

import os
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Path to GTZAN dataset
DATASET_PATH = "genres/"

genres = os.listdir(DATASET_PATH)
X, y = [], []

print(" Extracting features from dataset...")

# Extract MFCC features from audio files
for genre in genres:
    genre_path = os.path.join(DATASET_PATH, genre)
    print(f"Processing genre: {genre}")
    for file in os.listdir(genre_path):
        if file.endswith((".wav", ".au")):  # supports both
            file_path = os.path.join(genre_path, file)
            audio, sr = librosa.load(file_path, duration=30)
            mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
            mfcc_scaled = np.mean(mfcc.T, axis=0)
            X.append(mfcc_scaled)
            y.append(genre)

# Convert to numpy arrays
X = np.array(X)
y = np.array(y)

# Encode labels
le = LabelEncoder()
y = to_categorical(le.fit_transform(y))

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Reshape for LSTM
X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1])
X_test = X_test.reshape(X_test.shape[0], 1, X_test.shape[1])

# LSTM Model
model = Sequential([
    LSTM(128, input_shape=(1, 40), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=['accuracy'])
model.summary()

print("\n Training the model...")
model.fit(X_train, y_train, epochs=40, batch_size=32, validation_data=(X_test, y_test))

# Evaluate
loss, accuracy = model.evaluate(X_test, y_test)
print(f"\n Model Accuracy: {accuracy * 100:.2f}%")

# Save model and label encoder
model.save("music_genre_lstm_model.h5")
np.save("label_classes.npy", le.classes_)

print("\n Model saved successfully as 'music_genre_lstm_model.h5'")
print(" Label classes saved as 'label_classes.npy'")


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'genres/'

In [ ]:
# pip install librosa numpy tensorflow

import numpy as np
import librosa
import os
from tensorflow.keras.models import load_model

# Load model and label encoder
model = load_model("music_genre_lstm_model.h5")
classes = np.load("label_classes.npy", allow_pickle=True)

def predict_genre(file_path):
    if not os.path.exists(file_path):
        print(" File not found. Please check the path.")
        return

    print(f"\n Loading audio file: {file_path}")
    audio, sr = librosa.load(file_path, duration=30)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    mfcc_scaled = np.mean(mfcc.T, axis=0)
    mfcc_scaled = mfcc_scaled.reshape(1, 1, 40)

    prediction = model.predict(mfcc_scaled)
    predicted_index = np.argmax(prediction)
    predicted_genre = classes[predicted_index]

    print(f" Predicted Genre: {predicted_genre}")
    print(f" Confidence: {prediction[0][predicted_index] * 100:.2f}%")

# ---------------------------------------------------------------------
# Ask user for audio file path
# ---------------------------------------------------------------------
file_path = input("\nEnter path of audio file to predict (e.g. test_song.au): ")
predict_genre(file_path)

# C:\Users\ganes\OneDrive\Desktop\dlp\exam\b3 music type\genres\hiphop\hiphop.00004.au
# C:\Users\ganes\OneDrive\Desktop\dlp\exam\b3 music type\genres\blues\blues.00013.au
# C:\Users\ganes\OneDrive\Desktop\dlp\exam\b3 music type\genres\classical\classical.00001.au
# C:\Users\ganes\OneDrive\Desktop\dlp\exam\b3 music type\genres\classical\classical.00001.au
